# 11（進階）序列預測：用 LSTM / CNN 預測疫情曲線

Ch11 前半段我們用 PyTorch 訓練了一個分類模型，在 280 筆資料上和 sklearn 打成平手。
這裡我們換一個更適合深度學習發揮的任務：**序列預測（sequence forecasting）**。

流程：**合成登革熱 × 氣溫序列 → 時間窗口化 → 時間切分 train/val/test → LSTM → 1D-CNN → 與 naive 基準比較**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 為什麼序列預測不一樣？

前面章節（Ch03–Ch10）大多是「橫斷面」問題：每一列資料互相獨立，用今天的年齡、共病、暴露史去預測今天的結果。

**序列預測不同**：資料有時間順序，今天的病例數和昨天、前天……都有關聯（自相關）。而且流行病學裡常常有「領先指標」——某個變數會比病例數更早出現變化，等於提前預警。

### 本節情境（教學合成資料）

這裡**不使用**松柏護理之家的 Legionella 資料（那組資料沒有足夠長的每日序列），改用一組**合成的「登革熱 × 氣溫」每日序列**：

- **氣溫**：有季節性波動、每天有隨機噪音，是**已知、可提前取得**的領先指標
- **病例數**：受「7 天前的氣溫（越熱，病媒蚊活動越旺盛）」和「前一天病例數（傳播的延續性）」共同驅動，外加每週通報節律和隨機噪音

> ⚠️ 這是**教學用的合成資料**，不是真實的登革熱監測數據——重點是學習「怎麼用序列模型抓住領先指標」的方法，不是這組參數本身有流行病學意義。

**任務**：用過去 21 天的 (病例數, 氣溫)，預測 **7 天後**的病例數。

In [ ]:
# --- Step 1：生成合成的「登革熱 × 氣溫」序列 ---
import pathlib

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup（避免中文標籤顯示為方框）--
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# -- 固定隨機種子，確保每次執行結果一致（可重現研究的基本要求，見 Ch13）--
torch.manual_seed(1)
np.random.seed(1)

# -- 資料產生流程（DGP, data generating process）--
n = 360                       # 一年多一點的每日資料
t = np.arange(n)

# 氣溫：季節性正弦波 + 每日隨機噪音（這是「已知、可提前取得」的領先指標）
temp = 24 + 7 * np.sin(2 * np.pi * (t - 30) / 365) + np.random.normal(0, 1.0, n)

# drive：氣溫超過 24 度的部分，代表「越熱、病媒蚊活動越旺盛」
drive = np.clip(temp - 24, 0, None)

# 病例數：受「前一天病例數」（傳播延續性）+「7 天前氣溫驅動力」+ 每週節律 + 隨機噪音影響
cases = np.zeros(n)
for i in range(n):
    lag = cases[i - 1] if i >= 1 else 0
    cases[i] = max(
        0,
        0.55 * lag                                   # 前一天病例數的延續性
        + 3.2 * (drive[i - 7] if i >= 7 else 0)       # 7 天前氣溫的延遲效應（領先指標）
        + 4 * np.sin(2 * np.pi * t[i] / 7)            # 每週通報節律
        + 6                                            # 基準病例數
        + np.random.normal(0, 2.0),                   # 隨機噪音
    )

print(f"資料長度：{n} 天")
print(f"病例數：mean={cases.mean():.1f}, min={cases.min():.1f}, max={cases.max():.1f}")
print(f"氣溫：mean={temp.mean():.1f}°C, min={temp.min():.1f}°C, max={temp.max():.1f}°C")

# -- 畫出兩條時間序列，肉眼看看氣溫和病例數的關係 --
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(t, cases, color="#D97757", label="每日病例數")
ax1.set_xlabel("天數（day index）")
ax1.set_ylabel("每日病例數", color="#D97757")
ax1.tick_params(axis="y", labelcolor="#D97757")

ax2 = ax1.twinx()
ax2.plot(t, temp, color="#6A9BCC", alpha=0.7, label="氣溫（°C）")
ax2.set_ylabel("氣溫（°C）", color="#6A9BCC")
ax2.tick_params(axis="y", labelcolor="#6A9BCC")
ax2.grid(False)

fig.suptitle("合成登革熱 × 氣溫序列（教學資料）")
fig.tight_layout()
plt.show()

print("\n→ 仔細看：氣溫的高峰，是不是比病例數的高峰早出現幾天？這就是「領先指標」。")

## 從「一整條時間線」切成「一堆訓練樣本」

神經網路要吃固定形狀的輸入，不能直接餵一條 360 天長的序列。做法是**滑動窗口（sliding window）**：

- 每個樣本：過去 **L=21 天**的 (病例數, 氣溫) 當輸入 X，**7 天後（H=7）**的病例數當標籤 y
- 窗口往前移一天，就是下一個樣本——所以相鄰樣本會高度重疊，這是時間序列資料的常態

### 時間切分，不能用隨機切分！

Ch10 用 `train_test_split(shuffle=True)` 沒問題，因為那些資料列互相獨立。**序列資料絕對不能隨機切**——如果把未來的窗口混進訓練集，模型等於「看過答案」，離線表現會嚴重高估。

正確做法是**按時間切**：

1. 最前面 300 天當「訓練區間」，最後 60 天當**測試集**（模型從沒看過）
2. 訓練區間裡，再挖出**最後 40 天當驗證集（validation）**，用來做早停法——一樣是按時間切，不 shuffle
3. **標準化（normalize）用的平均數、標準差，只能用訓練區間（前 300 天）的統計量**，不能偷看測試集——這是避免 data leakage 的基本原則（呼應 Ch10 的 train/val 紀律）

In [ ]:
# --- Step 2：滑動窗口 + 時間切分 train/val/test + 只用訓練統計量標準化 ---
H = 7   # 要預測「幾天後」的病例數
L = 21  # 每個樣本回看「幾天」的歷史

# 把兩個特徵疊在一起：欄位 0 = cases，欄位 1 = temp
feats = np.stack([cases, temp], axis=1).astype(np.float32)

# split：前 300 天當「訓練區間」（train+val 都從這裡切），最後 60 天是完全沒看過的測試集
split = n - 60

# 標準化統計量只能用訓練區間算，不能偷看測試集（否則會有 data leakage）
mu = feats[:split].mean(axis=0)
sd = feats[:split].std(axis=0)
z = (feats - mu) / sd


def windows(s, e):
    """把標準化後的序列切成滑動窗口樣本。

    對每個 i in [s, e-H)：
      X = z[i-L : i]      → 過去 L 天的 (cases, temp)
      y = z[i+H-1, 0]     → H 天後的病例數（只取 cases 欄）

    回傳 (X 陣列, y 陣列, 對應的原始天數索引)，索引之後可以用來反標準化、對照真實病例數。
    """
    Xs, ys, idxs = [], [], []
    for i in range(s, e - H):
        Xs.append(z[i - L:i])
        ys.append(z[i + H - 1, 0])
        idxs.append(i + H - 1)
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32), np.array(idxs)


# 訓練區間（前 300 天）產生的全部窗口
Xtr_full, ytr_full, idxtr_full = windows(L, split)

# 時間切分：訓練區間裡，最後 40 天當驗證集（不 shuffle，按時間切）
val_start = split - 40
train_mask = idxtr_full < val_start
Xtr, ytr = Xtr_full[train_mask], ytr_full[train_mask]
Xval, yval = Xtr_full[~train_mask], ytr_full[~train_mask]

# 測試集：最後 60 天（模型從沒看過，標準化統計量也沒用到它）
Xte, yte, idxte = windows(split - H + 1, n + 1)

# 轉成 tensor
Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)
Xval_t, yval_t = torch.tensor(Xval), torch.tensor(yval).unsqueeze(1)
Xtrfull_t, ytrfull_t = torch.tensor(Xtr_full), torch.tensor(ytr_full).unsqueeze(1)
Xte_t, yte_t = torch.tensor(Xte), torch.tensor(yte).unsqueeze(1)

print(f"train：{Xtr.shape[0]:>3d} 個窗口（day {idxtr_full[train_mask].min()}–{idxtr_full[train_mask].max()}）")
print(f"val  ：{Xval.shape[0]:>3d} 個窗口（day {idxtr_full[~train_mask].min()}–{idxtr_full[~train_mask].max()}）")
print(f"test ：{Xte.shape[0]:>3d} 個窗口（day {idxte.min()}–{idxte.max()}）")
print(f"每個窗口形狀：{Xtr.shape[1:]} = (L={L} 天, 2 個特徵)")

## LSTM = 有記憶的偵探

LSTM（Long Short-Term Memory）內部有一組「記憶細胞」，會邊看序列邊決定「這個資訊要記住，還是忘掉」。

用偵探辦案來比喻：LSTM 讀完 21 天的線索後，腦中留下的不是最後一天的片段記憶，而是**整段案情的摘要**——「病例數這週是不是一直在爬升？氣溫是不是連續偏高？」——再根據這個摘要做出 7 天後的預測。

### 早停法（early stopping）怎麼加進來？

訓練迴圈裡，我們同時用**驗證集**監控 val MAE：

1. 先讓模型暖身訓練一段時間（前 150 個 epoch），這段時間 val 表現本來就會震盪，太早看容易被雜訊騙走
2. 暖身結束後才開始「patience 倒數」：val MAE 連續 30 個 epoch 都沒有進步，就停止訓練，並還原到「表現最好的那個 epoch」
3. 早停選出「該訓練幾個 epoch」之後，我們會在**全部**訓練前資料（train+val 合併）上，用同樣的 epoch 數重新訓練一次最終模型——因為序列預測中，最貼近測試期的那幾十天資料最有參考價值，不應該被永遠排除在正式模型之外

In [ ]:
# --- Step 3：LSTM 模型 + 訓練迴圈（含早停法） ---

class LSTMModel(nn.Module):
    """用 LSTM 讀過去 L 天的 (cases, temp)，輸出對 H 天後病例數的預測。"""

    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=32, batch_first=True)  # 2 個輸入特徵：cases, temp
        self.fc = nn.Linear(32, 1)  # 把最後的隱藏狀態轉成一個預測值

    def forward(self, x):
        out, _ = self.lstm(x)      # out 形狀：(batch, L, 32)，每個時間點都有一個隱藏狀態
        last_step = out[:, -1, :]  # 只取「看完整段序列後」最後一個時間點的隱藏狀態
        return self.fc(last_step)


def train_with_early_stopping(model_cls, seed=1, max_epochs=200, patience=30, warmup=150):
    """訓練迴圈 + 早停法（early stopping），回傳「表現最好的 epoch 編號」。

    做法：
      1. 每個 epoch 用「訓練集」更新權重（Adam + MSELoss）
      2. 每個 epoch 用「驗證集」算 val MAE，監控模型有沒有真的學到東西（而不是背答案）
      3. warmup 期間（前 150 個 epoch）先讓模型穩定訓練，不做早停判斷
      4. warmup 結束後才開始算 patience：val MAE 連續 30 個 epoch 沒有進步就停止
      5. 回傳最佳 epoch 編號——之後會在全部訓練資料上，用這個 epoch 數重新訓練一次正式模型
    """
    torch.manual_seed(seed)
    model = model_cls()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.008)
    loss_fn = nn.MSELoss()

    best_val_mae = float("inf")
    counter = 0
    best_epoch = warmup - 1

    for epoch in range(max_epochs):
        # -- 訓練一步 --
        model.train()
        optimizer.zero_grad()
        pred = model(Xtr_t)
        loss = loss_fn(pred, ytr_t)
        loss.backward()
        optimizer.step()

        # -- 用驗證集監控 --
        model.eval()
        with torch.no_grad():
            val_pred = model(Xval_t)
            val_mae = torch.mean(torch.abs(val_pred - yval_t)).item()

        if epoch < warmup:
            continue  # 暖身期：只訓練，不做早停判斷

        if val_mae < best_val_mae:
            best_val_mae, counter, best_epoch = val_mae, 0, epoch
        else:
            counter += 1
        if counter >= patience:
            print(f"  early stopping at epoch {epoch}（最佳 epoch = {best_epoch}）")
            break

    return best_epoch, best_val_mae


def refit_on_full(model_cls, n_epochs, seed=1):
    """用早停選出的 epoch 數，在『全部』測試前資料（train+val）上重新訓練最終模型。

    為什麼要多這一步？因為序列預測中，最貼近測試期的那幾十天資訊最寶貴——
    如果正式模型只用扣掉驗證集之後的訓練子集去配適，等於白白丟掉了最新的資訊。
    早停只負責幫我們決定「該訓練幾個 epoch」，決定好之後，
    在全部可用資料上用同樣的 epoch 數重新訓練一次，這是預測實務中常見的做法。
    """
    torch.manual_seed(seed)
    model = model_cls()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.008)
    loss_fn = nn.MSELoss()
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(Xtrfull_t)
        loss = loss_fn(pred, ytrfull_t)
        loss.backward()
        optimizer.step()
    return model


best_epoch_lstm, val_mae_lstm = train_with_early_stopping(LSTMModel)
print(f"LSTM：validation 選到的最佳 epoch = {best_epoch_lstm}（val MAE，標準化尺度 = {val_mae_lstm:.4f}）")

lstm_model = refit_on_full(LSTMModel, best_epoch_lstm + 1)
print(f"LSTM：已在全部訓練資料（train+val，共 {len(Xtr_full)} 個窗口）上重新訓練 {best_epoch_lstm + 1} 個 epoch")

## 1D-CNN = 看局部指紋

CNN（卷積神經網路）不像 LSTM 一路往下記，而是用一個很小的滑動視窗（kernel，這裡寬度設 3）掃過整段序列，專門抓**局部的形狀特徵**——例如「連續 3 天病例數在漲」、「氣溫這幾天忽然變高」——像在找指紋一樣，不管這個形狀出現在窗口的第幾天，只要出現就能被抓到。

兩層卷積疊起來，就能把「局部形狀」組合成更複雜的模式，最後攤平（flatten）接一層線性層輸出預測值。訓練方式和 LSTM 完全一樣：一樣用 Step 3 定義好的 `train_with_early_stopping()` 和 `refit_on_full()`，只是把模型換成 CNN。

In [ ]:
# --- Step 4：1D-CNN 模型 + 訓練（沿用 Step 3 的早停法函式） ---

class CNNModel(nn.Module):
    """兩層 1D 卷積抓局部形狀，攤平後接一層線性層輸出預測值。"""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=24, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(in_channels=24, out_channels=24, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.fc = nn.Linear(24 * L, 1)

    def forward(self, x):
        # Conv1d 預期輸入形狀是 (batch, channels, 序列長度)，
        # 但我們的 x 是 (batch, 序列長度 L, 特徵數 2)，所以要先轉置
        x = x.transpose(1, 2)
        return self.fc(self.net(x))


best_epoch_cnn, val_mae_cnn = train_with_early_stopping(CNNModel)
print(f"CNN：validation 選到的最佳 epoch = {best_epoch_cnn}（val MAE，標準化尺度 = {val_mae_cnn:.4f}）")

cnn_model = refit_on_full(CNNModel, best_epoch_cnn + 1)
print(f"CNN：已在全部訓練資料（train+val，共 {len(Xtr_full)} 個窗口）上重新訓練 {best_epoch_cnn + 1} 個 epoch")

## 怎麼知道模型有沒有用？——先打敗「什麼都不做」

單看 MAE 的絕對數字沒有意義，要跟一個**誠實的最低標準**比較：

- **Persistence（naive）基準**：直接假設「7 天後的病例數 = 現在最後已知的病例數」，完全不用任何模型、任何氣溫資訊
- 如果 LSTM / CNN 贏不過這個基準，代表模型的複雜度是白費的——這是評估序列預測最重要的第一道關卡

三個評估指標：

| 指標 | 意思 |
|------|------|
| MAE（Mean Absolute Error） | 平均誤差幾個病例數，最直覺 |
| RMSE（Root Mean Squared Error） | 對大誤差更敏感，一次離群值會被放大 |
| MAPE（Mean Absolute Percentage Error） | 誤差佔真實值的百分比，方便跨情境比較 |

In [ ]:
# --- Step 5：Persistence 基準 + 結果比較表 + 測試窗口預測圖 ---

def inverse_transform_cases(z_pred):
    """把標準化後的預測值還原成真實病例數尺度。"""
    return z_pred * sd[0] + mu[0]


def mae_score(pred, idxs):
    return np.mean(np.abs(pred - cases[idxs]))


def rmse_score(pred, idxs):
    return np.sqrt(np.mean((pred - cases[idxs]) ** 2))


def mape_score(pred, idxs):
    true = cases[idxs]
    return np.mean(np.abs(pred - true) / np.maximum(true, 1e-6)) * 100


# 模型預測（還原成病例數尺度）
with torch.no_grad():
    lstm_pred = inverse_transform_cases(lstm_model(Xte_t).squeeze(1).numpy())
    cnn_pred = inverse_transform_cases(cnn_model(Xte_t).squeeze(1).numpy())

# Persistence（naive）基準：7 天後病例數 = 現在最後已知的病例數，完全不用模型
persistence_pred = np.array([cases[j - H] for j in idxte])

results = {
    "Persistence（naive 基準）": persistence_pred,
    "LSTM": lstm_pred,
    "1D-CNN": cnn_pred,
}

print(f"{'模型':<24}{'MAE':>8}{'RMSE':>8}{'MAPE(%)':>10}")
print("-" * 50)
for name, pred in results.items():
    print(f"{name:<24}{mae_score(pred, idxte):>8.3f}{rmse_score(pred, idxte):>8.3f}{mape_score(pred, idxte):>10.2f}")

lstm_beats = mae_score(lstm_pred, idxte) < mae_score(persistence_pred, idxte)
cnn_beats = mae_score(cnn_pred, idxte) < mae_score(persistence_pred, idxte)
print(f"\nLSTM 贏過 persistence？ {lstm_beats}")
print(f"CNN  贏過 persistence？ {cnn_beats}")

# -- 畫出測試窗口的預測 vs 真實病例數 --
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(idxte, cases[idxte], label="真實病例數", color="#1A1A1A", linewidth=2)
ax.plot(idxte, persistence_pred, label="Persistence（naive）", color="#6B6B6B", linestyle="--")
ax.plot(idxte, lstm_pred, label="LSTM", color="#D97757")
ax.plot(idxte, cnn_pred, label="1D-CNN", color="#6A9BCC")
ax.set_xlabel("天數（day index）")
ax.set_ylabel("每日新增病例數")
ax.set_title("測試窗口：7 天後病例數預測 —— 真實值 vs 各模型")
ax.legend()
plt.tight_layout()
plt.show()

## 誠實的結論

本節的 LSTM / CNN 能打敗 persistence 基準，**不是因為深度學習天生比較強**，而是因為：

1. **它們用到了氣溫這個領先指標**——persistence 只看得到病例數本身，完全不知道 7 天前氣溫已經升高、7 天後病例數大概率會跟著漲
2. **DGP 裡有非線性、有時間延遲的交互作用**（7 天前氣溫驅動今天病例數的關係），這正是神經網路擅長抓的模式

### 什麼時候 naive 基準很難打敗？

如果曲線是**單純、平滑的單變量序列**（沒有額外的領先指標、沒有明顯的非線性延遲），persistence 或簡單的移動平均往往已經很強，複雜模型不見得能顯著超越——這時候多加一層 LSTM/CNN 可能只是徒增複雜度和過擬合風險。實務上遇到序列預測，**永遠要先跑一個 naive 基準**，再決定要不要上模型。

### DL 還是需要「足夠的歷史」

這裡的窗口長度（L=21 天）、訓練樣本數（272 個時間窗口）都遠比 Ch10/Ch11 前半段的 280 筆橫斷面資料更「巧婦難為無米之炊」——序列模型天生需要更長的歷史累積才能學到穩定的模式。真實世界的疫情監測序列如果只有幾十天，深度學習很可能不會有優勢。

下一章（Ch12），我們回到分類問題本身，但換一個角度問：淋浴「導致」感染，還是只是「相關」？→ 因果推論。